Para drift TODOS LOS TIPOS

(Cambios lentos → ventanas grandes → métricas deben ser menos sensibles)

psi:         0.25
ks:          0.15
wasserstein: 0.6 * std(ref)

# Drift Evaluation — Episodios (todas las clases de drift)

Notebook para evaluar performance de **ventana, estrategia y métrica estadística**
contra los episodios manuales etiquetados (abruptos, graduales u otros).


## 1. Imports

Reutilizamos exactamente el mismo backend (`Funciones_Drift.py`)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util
import plotly.express as px

plt.style.use("seaborn-v0_8")

spec = importlib.util.spec_from_file_location(
    "funciones_drift",
    "../Analisis/Funciones_Drift.py"
)
funciones_drift = importlib.util.module_from_spec(spec)
spec.loader.exec_module(funciones_drift)

ref_decay_prefix_mass  = funciones_drift.ref_decay_prefix_mass
ref_golden             = funciones_drift.ref_golden
ref_seasonal           = funciones_drift.ref_seasonal
score_numeric_series   = funciones_drift._score_numeric_series

## 2. Carga de datos y episodios manuales

In [2]:
SERIES_PATH = Path("synthetic_data/synthetic_plant.csv")
LABELS_PATH = Path("synthetic_data/synthetic_plant_events.csv")


df_raw = pd.read_csv(SERIES_PATH)

if "date_time" not in df_raw.columns:
    raise ValueError(
        "El DataFrame debe tener una columna de tiempo llamada 'date_time'.\n"
        "Por diseño del pipeline final esto se documentará en el README."
    )

df_raw["date_time"] = pd.to_datetime(df_raw["date_time"], errors="coerce")
df_raw = (
    df_raw
    .dropna(subset=["date_time"])
    .sort_values("date_time")
    .set_index("date_time")
)

df = df_raw.select_dtypes(include="number").copy()
assert not df.empty, "No hay columnas numéricas en la serie sintética."

t_min, t_max = df.index.min(), df.index.max()
print(f"Rango temporal de la serie: {t_min}  →  {t_max}")
print("N° de filas:", len(df), " | N° de variables numéricas:", df.shape[1])

events = pd.read_csv(LABELS_PATH)
events["date_time"] = pd.to_datetime(events["date_time"], errors="coerce")
events = (
    events
    .dropna(subset=["date_time", "variable", "event"])
    .assign(event=lambda s: s["event"].str.lower().str.strip())
    .query("event in ['start','end']")
    .sort_values(["variable", "date_time"])
    .reset_index(drop=True)
)

def events_to_intervals(ev: pd.DataFrame) -> pd.DataFrame:
    has_type = "drift_type" in ev.columns
    rows = []
    for var, g in ev.groupby("variable", sort=True):
        open_t = None
        open_type = None
        for _, r in g.iterrows():
            evt = str(r["event"]).lower()
            dt_val = r["drift_type"] if has_type else "unknown"

            if evt == "start":
                open_t = r["date_time"]
                open_type = dt_val
            elif evt == "end" and open_t is not None and r["date_time"] > open_t:
                rows.append({
                    "variable": var,
                    "manual_start": open_t,
                    "manual_end": r["date_time"],
                    "drift_type": open_type,
                })
                open_t = None
                open_type = None
    return pd.DataFrame(rows)

intervals_manual_all = events_to_intervals(events)
print("Total episodios manuales:", len(intervals_manual_all))

# ✅ NUEVO: usamos TODOS los episodios manuales (abrupt + gradual + otros)
intervals_manual = (
    intervals_manual_all
    .sort_values(["variable", "manual_start"])
    .reset_index(drop=True)
)
if "drift_type" in intervals_manual.columns:
    print("\nDistribución por drift_type en etiquetas manuales:")
    print(intervals_manual["drift_type"].value_counts())

Rango temporal de la serie: 2025-01-01 00:00:00  →  2025-01-20 23:59:00
N° de filas: 28800  | N° de variables numéricas: 10
Total episodios manuales: 23

Distribución por drift_type en etiquetas manuales:
drift_type
gradual    15
abrupt      8
Name: count, dtype: int64


## 3. Detección de drift

In [3]:
def run_drift_for_strategy_multi_metric(
    df: pd.DataFrame,
    window: str,
    strategy: str,
    metrics: tuple = ("psi", "ks", "wasserstein", "mannwhitney"),
    thresholds: dict | None = None,
    min_points: int = 5,
):
    """
    Ejecuta detección de drift para una estrategia dada y *varias* métricas numéricas.

    - NO hay referencia congelada.
    - En cada ventana se construye la referencia según `strategy` usando TODO el historial hasta t0.
    - Para cada métrica en `metrics`:
        stat_value = score_numeric_series(ref, cur, metric)
        drift_flag = stat_value >= threshold_métrica
    - Un episodio es una secuencia contigua de ventanas con drift_flag=True.
    - Cuando la métrica baja del umbral, se cierra el episodio y listo; la referencia
      en pasos futuros ya incorpora el nuevo nivel estable.

    `thresholds`: dict opcional {metric_name: valor}. Si no se pasa, usa los defaults
    coherentes con tu Funciones_Drift:

        psi         ~ 0.2
        ks          ~ 0.15
        wasserstein -> 0.5 * std(ref) si no se fija
    """

    # Defaults coherentes con _dispatch_build_metrics
    default_thr = {"psi": 0.2, "ks": 0.15, "wasserstein": np.nan}
    thresholds = thresholds or {}

    w = pd.to_timedelta(window)
    t_min, t_max = df.index.min(), df.index.max()
    t_ends = pd.date_range(t_min + w, t_max, freq=window)

    variables = list(df.columns)

    # Estado por (métrica, variable) sólo para numerar episodios contiguos
    state = {
        metric: {var: "NORMAL" for var in variables}
        for metric in metrics
    }
    current_episode = {
        metric: {var: 0 for var in variables}
        for metric in metrics
    }

    rows = []

    for t_end in t_ends:
        t0 = t_end - w

        df_hist = df.loc[: t0 - pd.Timedelta(microseconds=1)]
        df_cur  = df.loc[t0:t_end]

        if df_hist.empty or df_cur.empty:
            continue

        # Referencia según la estrategia (SIEMPRE recalculada)
        if strategy == "decay":
            ref_global = ref_decay_prefix_mass(df_hist, now=t_end)
        elif strategy == "golden":
            ref_global = ref_golden(df_hist)
        elif strategy == "seasonal":
            ref_global = ref_seasonal(df_hist, current_end=t_end)
        else:
            raise ValueError(f"Estrategia desconocida: {strategy}")

        if ref_global is None or ref_global.empty:
            ref_global = df_hist

        for var in variables:
            cur_series = df_cur[var].dropna()

            if cur_series.size < min_points:
                # Sin datos suficientes: logueamos filas sin drift (no cambiamos estado)
                for metric_name in metrics:
                    rows.append({
                        "variable": var,
                        "strategy": strategy,
                        "window": window,
                        "metric": metric_name,
                        "t0": t0,
                        "t1": t_end,
                        "drift_flag": False,
                        "episode_id": np.nan,
                        "stat_value": None,
                        "threshold": None,
                        "state": state[metric_name][var],
                    })
                continue

            # Referencia para esta variable
            if var in ref_global.columns:
                ref_series = ref_global[var].dropna()
            else:
                ref_series = df_hist[var].dropna()

            for metric_name in metrics:
                base_thr = default_thr.get(metric_name, 0.2)
                thr = thresholds.get(metric_name, base_thr)

                if ref_series.empty:
                    stat_val = None
                    eff_thr = thr
                    drift_flag = False
                else:
                    # Usa exactamente la misma función que el pipeline
                    stat_val = score_numeric_series(ref_series, cur_series, metric_name)

                    # Para Wasserstein, si el threshold viene como NaN, lo adaptamos a la escala ref
                    eff_thr = thr
                    if metric_name == "wasserstein" and (eff_thr is None or (isinstance(eff_thr, float) and np.isnan(eff_thr))):
                        std_ref = pd.to_numeric(ref_series, errors="coerce").dropna().std()
                        eff_thr = float(std_ref) * 0.5 if pd.notna(std_ref) else 0.5

                    if stat_val is None or np.isnan(stat_val):
                        drift_flag = False
                    else:
                        drift_flag = bool(stat_val >= eff_thr)

                # Actualizar estado/episodios (sin referencia congelada)
                if drift_flag:
                    if state[metric_name][var] == "NORMAL":
                        current_episode[metric_name][var] += 1
                        state[metric_name][var] = "DRIFT"
                else:
                    if state[metric_name][var] == "DRIFT":
                        state[metric_name][var] = "NORMAL"

                rows.append({
                    "variable": var,
                    "strategy": strategy,
                    "window": window,
                    "metric": metric_name,
                    "t0": t0,
                    "t1": t_end,
                    "drift_flag": drift_flag,
                    "episode_id": (
                        current_episode[metric_name][var]
                        if drift_flag else np.nan
                    ),
                    "stat_value": stat_val,
                    "threshold": eff_thr,
                    "state": state[metric_name][var],
                })

    return pd.DataFrame(rows)


def run_drift_all_multi_metric(
    df: pd.DataFrame,
    windows=("6H","12H","24H","48H"),
    strategies=("decay","golden","seasonal"),
    metrics: tuple = ("psi", "ks", "wasserstein"),
    thresholds: dict | None = None,
    min_points: int = 5,
):
    """Runner multi-ventana, multi-estrategia y multi-métrica."""
    all_frames = []
    for win in windows:
        for strat in strategies:
            print(f"[DRIFT] window={win}, strategy={strat}")
            dfw = run_drift_for_strategy_multi_metric(
                df=df,
                window=win,
                strategy=strat,
                metrics=metrics,
                thresholds=thresholds,
                min_points=min_points,
            )
            all_frames.append(dfw)

    if not all_frames:
        return pd.DataFrame()
    return pd.concat(all_frames, ignore_index=True)

In [4]:
# Configuración de evaluación (todas las clases de drift)
EVAL_WINDOWS = ["6H", "12H", "24H", "36H", "48H"]
STRATEGIES   = ["decay","golden","seasonal"]
METRICS      = ("psi", "ks", "wasserstein")

METRIC_THRESHOLDS = {
    "psi": 0.30,         # antes 0.2
    "ks":  0.20,         # antes 0.15
    "wasserstein": np.nan,  # se ajusta a 0.5 * std(ref) automáticamente
}

df_windows = run_drift_all_multi_metric(
    df=df,
    windows=EVAL_WINDOWS,
    strategies=STRATEGIES,
    metrics=METRICS,
    thresholds=METRIC_THRESHOLDS,
    min_points=60,
)

[DRIFT] window=6H, strategy=decay
[DRIFT] window=6H, strategy=golden
[DRIFT] window=6H, strategy=seasonal
[DRIFT] window=12H, strategy=decay
[DRIFT] window=12H, strategy=golden
[DRIFT] window=12H, strategy=seasonal
[DRIFT] window=24H, strategy=decay
[DRIFT] window=24H, strategy=golden
[DRIFT] window=24H, strategy=seasonal
[DRIFT] window=36H, strategy=decay
[DRIFT] window=36H, strategy=golden
[DRIFT] window=36H, strategy=seasonal
[DRIFT] window=48H, strategy=decay
[DRIFT] window=48H, strategy=golden
[DRIFT] window=48H, strategy=seasonal


## 4. Compactar ventanas en episodios automáticos (TODOS LOS TIPOS)

Mismo procedimiento que en el notebook de ABRUPT.

In [5]:
def windows_to_episodes_multi_metric(
    df_windows: pd.DataFrame,
    min_windows: int = 1,   # 🔹 por defecto NO filtra episodios cortos
) -> pd.DataFrame:
    """
    Compacta secuencias de ventanas con drift_flag=True en episodios automáticos.

    - Agrupa por (window, strategy, metric, variable, episode_id).
    - Cada grupo representa un episodio candidato.
    - Si el episodio tiene menos de `min_windows` ventanas, se descarta.
      Ojo: con min_windows > 1 puedes perder drifts MUY cortos (1 sola ventana).
    """
    dfw = df_windows.copy()
    dfw = dfw[dfw["drift_flag"] == True].dropna(subset=["episode_id"])
    if dfw.empty:
        return pd.DataFrame(columns=[
            "window","strategy","metric","variable","episode_id",
            "seg_start","seg_end","seg_length","stat_max","n_windows"
        ])

    rows = []
    for keys, sub in dfw.groupby(
        ["window","strategy","metric","variable","episode_id"],
        dropna=False
    ):
        win, strat, metric, var, eid = keys
        sub = sub.sort_values("t0")

        n_windows = len(sub)
        if n_windows < min_windows:
            continue  # demasiado corto según el criterio elegido

        seg_start = sub["t0"].min()
        seg_end   = sub["t1"].max()
        stat_max  = sub["stat_value"].max()

        rows.append({
            "window": win,
            "strategy": strat,
            "metric": metric,
            "variable": var,
            "episode_id": int(eid),
            "seg_start": seg_start,
            "seg_end": seg_end,
            "seg_length": seg_end - seg_start,
            "stat_max": stat_max,
            "n_windows": n_windows,
        })

    return pd.DataFrame(rows)

df_episodes_auto = windows_to_episodes_multi_metric(df_windows, min_windows=1)
print("Episodios automáticos:", len(df_episodes_auto))
display(df_episodes_auto.head())

Episodios automáticos: 1026


,window,strategy,metric,variable,episode_id,seg_start,seg_end,seg_length,stat_max,n_windows
0,12H,decay,ks,var_1,1,2025-01-13 12:00:00,2025-01-20 12:00:00,7 days 00:00:00,0.905897,14
1,12H,decay,ks,var_10,1,2025-01-06 12:00:00,2025-01-20 12:00:00,14 days 00:00:00,0.906784,28
2,12H,decay,ks,var_2,1,2025-01-16 00:00:00,2025-01-20 12:00:00,4 days 12:00:00,0.938501,9
3,12H,decay,ks,var_3,1,2025-01-15 00:00:00,2025-01-20 12:00:00,5 days 12:00:00,0.880733,11
4,12H,decay,ks,var_4,1,2025-01-11 00:00:00,2025-01-20 12:00:00,9 days 12:00:00,0.974401,19


## 5. Evaluación vs episodios manuales

In [6]:
def evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
):
    if df_episodes_auto.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    man = intervals_manual.copy()
    total_days = max((t_max - t_min).total_seconds() / (3600 * 24), 1e-9)

    results = []
    marks_manual_all = []
    marks_auto_all   = []

    for (win, strat, metric), auto_sub in df_episodes_auto.groupby(
        ["window", "strategy", "metric"], dropna=False
    ):
        vars_in_auto = sorted(auto_sub["variable"].unique())
        man_sub = man[man["variable"].isin(vars_in_auto)].copy()
        if man_sub.empty and auto_sub.empty:
            continue

        manual_matches = []
        coverage_vals  = []
        delay_vals     = []

        for _, mrow in man_sub.iterrows():
            v  = mrow["variable"]
            ms = mrow["manual_start"]
            me = mrow["manual_end"]

            rel_auto = auto_sub[auto_sub["variable"] == v]

            total_overlap = pd.Timedelta(0)
            first_det = None

            for _, arow in rel_auto.iterrows():
                as_ = arow["seg_start"]
                ae  = arow["seg_end"]

                start = max(ms, as_)
                end   = min(me, ae)
                if end > start:
                    total_overlap += (end - start)

                    det_candidate = as_
                    if det_candidate < ms:
                        det_candidate = ms
                    if first_det is None or det_candidate < first_det:
                        first_det = det_candidate

            matched = total_overlap > pd.Timedelta(0)
            manual_matches.append(bool(matched))

            dur = me - ms
            if dur.total_seconds() > 0:
                cov_val = total_overlap.total_seconds() / dur.total_seconds()
            else:
                cov_val = 0.0
            coverage_vals.append(cov_val)

            if first_det is None:
                delay_vals.append(np.nan)
            else:
                                # 🔹 Ajuste: consideramos que la detección efectiva ocurre 6 horas más tarde
                first_det_shifted = first_det + pd.Timedelta(hours=6)
                delay = (first_det_shifted - ms).total_seconds() / 3600.0
                if delay < 0:
                    delay = 0.0
                delay_vals.append(delay)

        man_sub["matched_auto"] = manual_matches
        man_sub["coverage"]     = coverage_vals
        man_sub["delay_hours"]  = delay_vals

        TP = int(man_sub["matched_auto"].sum()) if not man_sub.empty else 0
        FN = int((~man_sub["matched_auto"]).sum()) if not man_sub.empty else 0

        auto_sub = auto_sub.copy()
        auto_matches = []
        for _, arow in auto_sub.iterrows():
            v  = arow["variable"]
            as_ = arow["seg_start"]
            ae  = arow["seg_end"]

            overlap = (
                (man_sub["variable"] == v) &
                ~(man_sub["manual_end"] < as_) &
                ~(man_sub["manual_start"] > ae)
            ).any()
            auto_matches.append(overlap)

        auto_sub["matched_manual"] = auto_matches
        FP = int((~auto_sub["matched_manual"]).sum()) if not auto_sub.empty else 0

        prec = TP / (TP + FP) if (TP + FP) > 0 else np.nan
        rec  = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        if np.isnan(prec) or np.isnan(rec) or (prec + rec) == 0:
            f1 = np.nan
        else:
            f1 = 2 * prec * rec / (prec + rec)

        coverage_mean   = float(np.nanmean(coverage_vals))  if coverage_vals else np.nan
        coverage_median = float(np.nanmedian(coverage_vals)) if coverage_vals else np.nan

        delay_valid = [d for d in delay_vals if not np.isnan(d)]
        delay_mean_hours   = float(np.mean(delay_valid))   if delay_valid else np.nan
        delay_median_hours = float(np.median(delay_valid)) if delay_valid else np.nan

        false_alarms_per_day = FP / total_days

        if not man_sub.empty:
            man_sub["manual_len_sec"] = (
                man_sub["manual_end"] - man_sub["manual_start"]
            ).dt.total_seconds()
            manual_len_total_sec = man_sub["manual_len_sec"].sum()
        else:
            manual_len_total_sec = 0.0

        if not auto_sub.empty:
            auto_sub["auto_len_sec"] = (
                auto_sub["seg_end"] - auto_sub["seg_start"]
            ).dt.total_seconds()
            auto_len_total_sec = auto_sub["auto_len_sec"].sum()
        else:
            auto_len_total_sec = 0.0

        overlap_total_sec = 0.0
        if manual_len_total_sec > 0 and auto_len_total_sec > 0:
            for _, mrow in man_sub.iterrows():
                v  = mrow["variable"]
                ms = mrow["manual_start"]
                me = mrow["manual_end"]

                rel_auto = auto_sub[auto_sub["variable"] == v]
                for _, arow in rel_auto.iterrows():
                    as_ = arow["seg_start"]
                    ae  = arow["seg_end"]
                    start = max(ms, as_)
                    end   = min(me, ae)
                    if end > start:
                        overlap_total_sec += (end - start).total_seconds()

        if auto_len_total_sec > 0:
            prec_time = overlap_total_sec / auto_len_total_sec
        else:
            prec_time = np.nan

        if manual_len_total_sec > 0:
            rec_time = overlap_total_sec / manual_len_total_sec
        else:
            rec_time = np.nan

        if np.isnan(prec_time) or np.isnan(rec_time) or (prec_time + rec_time) == 0:
            f1_time = np.nan
        else:
            f1_time = 2 * prec_time * rec_time / (prec_time + rec_time)

        extra_time_sec = max(auto_len_total_sec - overlap_total_sec, 0.0)
        extra_hours = extra_time_sec / 3600.0 if extra_time_sec > 0 else 0.0

        if auto_len_total_sec > 0:
            extra_ratio_auto = extra_time_sec / auto_len_total_sec
        else:
            extra_ratio_auto = np.nan

        results.append({
            "window": win,
            "strategy": strat,
            "metric": metric,
            "TP_episodes": TP,
            "FP_episodes": FP,
            "FN_episodes": FN,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "manual_total_hours": manual_len_total_sec / 3600.0 if manual_len_total_sec > 0 else 0.0,
            "auto_total_hours": auto_len_total_sec / 3600.0 if auto_len_total_sec > 0 else 0.0,
            "overlap_hours": overlap_total_sec / 3600.0 if overlap_total_sec > 0 else 0.0,
            "Precision_time": prec_time,
            "Recall_time": rec_time,
            "F1_time": f1_time,
            "coverage_mean": coverage_mean,
            "coverage_median": coverage_median,
            "delay_mean_hours": delay_mean_hours,
            "delay_median_hours": delay_median_hours,
            "false_alarms_per_day": false_alarms_per_day,
            "extra_hours": extra_hours,
            "extra_ratio_auto": extra_ratio_auto,
        })

        man_sub["window"]   = win
        man_sub["strategy"] = strat
        man_sub["metric"]   = metric

        auto_sub["window"]   = win
        auto_sub["strategy"] = strat
        auto_sub["metric"]   = metric

        marks_manual_all.append(man_sub)
        marks_auto_all.append(auto_sub)

    eval_df = pd.DataFrame(results)
    manual_marked = (
        pd.concat(marks_manual_all, ignore_index=True)
        if marks_manual_all else pd.DataFrame()
    )
    auto_marked = (
        pd.concat(marks_auto_all, ignore_index=True)
        if marks_auto_all else pd.DataFrame()
    )

    return eval_df, manual_marked, auto_marked


eval_episodes_df, manual_marked, auto_marked = evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto=df_episodes_auto,
    intervals_manual=intervals_manual,
)

eval_by_type_df = pd.DataFrame()
summary_by_type = pd.DataFrame()

intervals_manual_norm = intervals_manual.copy()
intervals_manual_norm["drift_type_norm"] = (
    intervals_manual_norm["drift_type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

subsets = {
    "all": intervals_manual_norm,
    "gradual": intervals_manual_norm[intervals_manual_norm["drift_type_norm"] == "gradual"],
    "abrupt": intervals_manual_norm[intervals_manual_norm["drift_type_norm"] == "abrupt"],
}

eval_by_type_list = []

for label, inter_sub in subsets.items():
    if inter_sub.empty:
        continue

    inter_sub_clean = inter_sub.drop(columns=["drift_type_norm"])

    eval_sub, _, _ = evaluate_episodes_vs_manual_multi_metric(
        df_episodes_auto=df_episodes_auto,
        intervals_manual=inter_sub_clean,
    )
    if eval_sub.empty:
        continue

    eval_sub = eval_sub.copy()
    eval_sub["drift_group"] = label
    eval_by_type_list.append(eval_sub)

if eval_by_type_list:
    eval_by_type_df = pd.concat(eval_by_type_list, ignore_index=True)

    # Resumen simple: comparar gradual vs abrupto
    summary_by_type = (
        eval_by_type_df
        .groupby("drift_group")[["F1", "F1_time", "delay_mean_hours", "extra_ratio_auto"]]
        .mean()
        .reset_index()
    )

    print("Resumen por tipo de drift:")
    display(summary_by_type)

    print("\nEjemplo de filas crudas por tipo:")
    display(eval_by_type_df.head())

Resumen por tipo de drift:


,drift_group,F1,F1_time,delay_mean_hours,extra_ratio_auto
0,abrupt,0.433425,0.013378,6.020291,0.993252
1,all,0.827214,0.536137,7.246705,0.605617
2,gradual,0.785226,0.530022,7.787922,0.612366



Ejemplo de filas crudas por tipo:


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,Recall_time,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto,drift_group
0,12H,decay,ks,22,0,1,1.000000,0.956522,0.977778,790.366667,...,0.960019,0.557432,0.935184,1.000000,6.878788,6.000000,0.000000,1173.233333,0.607264,all
1,12H,decay,psi,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.960019,0.554985,0.935184,1.000000,6.878788,6.000000,0.050002,1185.233333,0.609688,all
2,12H,decay,wasserstein,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.960019,0.564902,0.935184,1.000000,6.878788,6.000000,0.050002,1137.233333,0.599807,all
3,12H,golden,ks,22,4,1,0.846154,0.956522,0.897959,790.366667,...,0.852115,0.668680,0.862166,0.981074,8.820455,6.000000,0.200007,550.516667,0.449769,all
4,12H,golden,psi,21,3,2,0.875000,0.913043,0.893617,790.366667,...,0.780482,0.635170,0.781735,0.905717,9.903968,6.816667,0.150005,535.133333,0.464525,all


In [7]:
eval_episodes_df = eval_episodes_df.copy()
eval_episodes_df["F1_time_filled"] = eval_episodes_df["F1_time"].fillna(0.0)
eval_episodes_df["Score_F1_combined"] = (
    0.85 * eval_episodes_df["F1"] + 0.15 * eval_episodes_df["F1_time_filled"])
display(eval_episodes_df.head())

,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto,F1_time_filled,Score_F1_combined
0,12H,decay,ks,22,0,1,1.000000,0.956522,0.977778,790.366667,...,0.557432,0.935184,1.000000,6.878788,6.000000,0.000000,1173.233333,0.607264,0.557432,0.914726
1,12H,decay,psi,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.554985,0.935184,1.000000,6.878788,6.000000,0.050002,1185.233333,0.609688,0.554985,0.896291
2,12H,decay,wasserstein,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.564902,0.935184,1.000000,6.878788,6.000000,0.050002,1137.233333,0.599807,0.564902,0.897779
3,12H,golden,ks,22,4,1,0.846154,0.956522,0.897959,790.366667,...,0.668680,0.862166,0.981074,8.820455,6.000000,0.200007,550.516667,0.449769,0.668680,0.863567
4,12H,golden,psi,21,3,2,0.875000,0.913043,0.893617,790.366667,...,0.635170,0.781735,0.905717,9.903968,6.816667,0.150005,535.133333,0.464525,0.635170,0.854850


## 6. Exportar CSVs de resultados (TODOS LOS TIPOS)

Guardamos en `synthetic_data/results_todos los tipos/`. 

In [8]:
OUTPUT_DIR = Path("synthetic_data/results")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

eval_path   = OUTPUT_DIR / "eval_episodes_by_window_strategy.csv"
manual_path = OUTPUT_DIR / "manual_marked_episodes.csv"
auto_path   = OUTPUT_DIR / "auto_marked_episodes.csv"

eval_episodes_df.to_csv(eval_path, index=False)
if not manual_marked.empty:
    manual_marked.to_csv(manual_path, index=False)
if not auto_marked.empty:
    auto_marked.to_csv(auto_path, index=False)

print("Guardados (GRADUAL):")
print(" -", eval_path)
if manual_marked is not None and not manual_marked.empty:
    print(" -", manual_path)
if auto_marked is not None and not auto_marked.empty:
    print(" -", auto_path)

METRICS_DIR = OUTPUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
print(f"📂 Directorio de métricas avanzadas: {METRICS_DIR.resolve()}")

has_cov_mean   = "coverage_mean" in eval_episodes_df.columns
has_delay_mean = "delay_mean_hours" in eval_episodes_df.columns
has_delay_med  = "delay_median_hours" in eval_episodes_df.columns
has_false_rate = "false_alarms_per_day" in eval_episodes_df.columns

quality_cols = ["Precision", "Recall", "F1"]
time_cols = [c for c in ["Precision_time", "Recall_time", "F1_time"]
             if c in eval_episodes_df.columns]
quality_cols.extend(time_cols)
if has_cov_mean:
    quality_cols.append("coverage_mean")

if "extra_ratio_auto" in eval_episodes_df.columns:
    quality_cols.append("extra_ratio_auto")

quality_group = ["metric", "strategy", "window"]

quality_overall = (
    eval_episodes_df
    .groupby(quality_group)[quality_cols]
    .mean()
    .reset_index()
    .sort_values(["metric", "window", "F1"], ascending=[True, True, False])
)
quality_overall.to_csv(METRICS_DIR / "quality_overall.csv", index=False)
print(f"✅ quality_overall.csv guardado ({len(quality_overall)} filas)")

if has_delay_mean or has_delay_med:
    delay_cols = []
    if has_delay_mean:
        delay_cols.append("delay_mean_hours")
    if has_delay_med:
        delay_cols.append("delay_median_hours")

    speed_overall = (
        eval_episodes_df
        .groupby(quality_group)[delay_cols]
        .mean()
        .reset_index()
        .sort_values(["metric", "window"], ascending=[True, True])
    )
    speed_overall.to_csv(METRICS_DIR / "speed_overall.csv", index=False)
    print(f"✅ speed_overall.csv guardado ({len(speed_overall)} filas)")
else:
    print("ℹ️ No hay columnas de delay → se omite speed_overall.csv")

if has_false_rate:
    stability_overall = (
        eval_episodes_df
        .groupby(quality_group)[["false_alarms_per_day"]]
        .mean()
        .reset_index()
        .sort_values(["metric", "window", "false_alarms_per_day"],
                     ascending=[True, True, True])
    )
    stability_overall.to_csv(METRICS_DIR / "stability_overall.csv", index=False)
    print(f"✅ stability_overall.csv guardado ({len(stability_overall)} filas)")
else:
    print("ℹ️ No se encontró 'false_alarms_per_day' → se omite stability_overall.csv")

if not eval_by_type_df.empty:
    eval_by_type_df.to_csv(METRICS_DIR / "quality_by_drift_type_raw.csv", index=False)
    if not summary_by_type.empty:
        summary_by_type.to_csv(METRICS_DIR / "quality_by_drift_type_summary.csv", index=False)
        print(f"✅ quality_by_drift_type_summary.csv guardado ({len(summary_by_type)} filas)")


print("🎯 Export de métricas terminado")

Guardados (GRADUAL):
 - synthetic_data\results\eval_episodes_by_window_strategy.csv
 - synthetic_data\results\manual_marked_episodes.csv
 - synthetic_data\results\auto_marked_episodes.csv
📂 Directorio de métricas avanzadas: C:\Users\frncc\OneDrive - Universidad Católica de Chile\Desktop\UC\2025-2\Proyecto de Grado\Proyecto-Grado\Drift Evaluation\synthetic_data\results\metrics
✅ quality_overall.csv guardado (45 filas)
✅ speed_overall.csv guardado (45 filas)
✅ stability_overall.csv guardado (45 filas)
✅ quality_by_drift_type_summary.csv guardado (3 filas)
🎯 Export de métricas terminado


## 7. Visualización 5×2 (Matplotlib) — TODOS LOS TIPOS

In [9]:
def plot_grid_episodes_matplotlib(
    df,
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
    strategy: str,
    window: str,
    metric: str = "psi",
    show_manual: bool = True,
):
    vars10 = list(df.columns)[:10]
    rows, cols = 5, 2

    subset_auto = df_episodes_auto[
        (df_episodes_auto["strategy"] == strategy) &
        (df_episodes_auto["window"] == window) &
        (df_episodes_auto["metric"] == metric)
    ]

    fig, axes = plt.subplots(rows, cols, figsize=(12, 16), sharex=True)
    axes = axes.flatten()

    for i, var in enumerate(vars10):
        ax = axes[i]
        s = df[var]

        ax.plot(s.index, s.values, linewidth=0.8)
        ax.set_title(var, fontsize=9)

        if show_manual:
            mans = intervals_manual[intervals_manual["variable"] == var]
            for _, m in mans.iterrows():
                ax.axvspan(m["manual_start"], m["manual_end"],
                           alpha=0.18, color="red")

        autos = subset_auto[subset_auto["variable"] == var]
        for _, a in autos.iterrows():
            ax.axvspan(a["seg_start"], a["seg_end"],
                       alpha=0.18, color="blue")

        ax.grid(True, alpha=0.3)

    for j in range(len(vars10), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        f"Episodios GRADUAL — strategy={strategy}, window={window}, metric={metric}\n"
        f"(rojo = manual, azul = auto)",
        fontsize=12
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    return fig


EXPORT_ROOT = OUTPUT_DIR / "plots"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

PLOT_WINDOWS     = EVAL_WINDOWS
PLOT_STRATEGIES  = STRATEGIES
PLOT_METRICS     = METRICS

print("Exportando combinaciones a PNG")

for win in PLOT_WINDOWS:
    win_dir = EXPORT_ROOT / f"window_{win}"
    win_dir.mkdir(parents=True, exist_ok=True)

    for strat in PLOT_STRATEGIES:
        for metric in PLOT_METRICS:
            subset_auto = df_episodes_auto[
                (df_episodes_auto["strategy"] == strat) &
                (df_episodes_auto["window"]   == win) &
                (df_episodes_auto["metric"]   == metric)
            ]
            if subset_auto.empty:
                continue

            print("\n==============================================")
            print(f"Ventana: {win} | Estrategia: {strat} | Métrica: {metric}")
            print("==============================================")

            fig = plot_grid_episodes_matplotlib(
                df=df,
                df_episodes_auto=df_episodes_auto,
                intervals_manual=intervals_manual,
                strategy=strat,
                window=win,
                metric=metric,
                show_manual=True
            )

            out_path = win_dir / f"episodes_{metric}_{strat}_{win}.png"
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)

            print(f"[OK] Exportado: {out_path}")

Exportando combinaciones a PNG

Ventana: 6H | Estrategia: decay | Métrica: psi
[OK] Exportado: synthetic_data\results\plots\window_6H\episodes_psi_decay_6H.png

Ventana: 6H | Estrategia: decay | Métrica: ks
[OK] Exportado: synthetic_data\results\plots\window_6H\episodes_ks_decay_6H.png

Ventana: 6H | Estrategia: decay | Métrica: wasserstein
[OK] Exportado: synthetic_data\results\plots\window_6H\episodes_wasserstein_decay_6H.png

Ventana: 6H | Estrategia: golden | Métrica: psi
[OK] Exportado: synthetic_data\results\plots\window_6H\episodes_psi_golden_6H.png

Ventana: 6H | Estrategia: golden | Métrica: ks
[OK] Exportado: synthetic_data\results\plots\window_6H\episodes_ks_golden_6H.png

Ventana: 6H | Estrategia: golden | Métrica: wasserstein
[OK] Exportado: synthetic_data\results\plots\window_6H\episodes_wasserstein_golden_6H.png

Ventana: 6H | Estrategia: seasonal | Métrica: psi
[OK] Exportado: synthetic_data\results\plots\window_6H\episodes_psi_seasonal_6H.png

Ventana: 6H | Estrategia: